# Burgers Equation: Shocks, Diffusion, and Nonlinear Transport

The **Burgers equation** is one of the simplest nonlinear partial differential equations that combines transport with diffusion:
$$
\partial_t u + u\,\partial_x u = \nu\,\partial_{xx} u, \qquad x \in [0,1],\; t > 0.
$$

It was introduced by Johannes Burgers (1948) as a model for turbulence and has since become a fundamental test-bed for understanding **shock formation**, **entropy conditions**, and the **viscosity limit** in conservation laws. It appears naturally in gas dynamics, traffic flow, cosmological large-scale structure formation, and the KPZ universality class in stochastic growth.

**Structure of the equation:**
- The nonlinear term $u\,\partial_x u = \frac{1}{2}\partial_x(u^2)$ is a **self-steepening transport**: regions where $u$ is larger move faster, so the profile inevitably folds, producing a **shock** in finite time when $\nu = 0$.
- The diffusion term $\nu\,\partial_{xx} u$ regularizes the shock into a smooth but steep transition layer of width $\mathcal{O}(\nu)$.

**The inviscid limit** $\nu \to 0^+$ selects the physically relevant **entropy solution** via the **Hopf–Lax formula**:
$$
u(x, t) = \frac{x - y^*(x,t)}{t}, \qquad \text{where }y^*(x,t) = \operatorname{argmin}_y \Bigl[\tfrac{(x-y)^2}{2t} + F_0(y)\Bigr],
$$
with $F_0(y) = \int_0^y u_0(s)\,ds$ the primitive of the initial condition.

This notebook constructs finite-difference solutions for several initial conditions, studies the effect of viscosity on shock formation, and provides interactive controls to explore the parameter space.

## Environment

Pure NumPy/Matplotlib implementation; `ipywidgets` for interactive parameter controls.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatLogSlider, Dropdown

plt.rcParams["figure.dpi"] = 120

## Finite-difference discretization

We discretize on a periodic domain $[0,1)$ with $n$ equally-spaced points $x_j = j/n$. The spatial derivatives use **centered finite differences**:
$$
\partial_x u_j \approx \frac{u_{j+1} - u_{j-1}}{2h}, \qquad
\partial_{xx} u_j \approx \frac{u_{j+1} - 2u_j + u_{j-1}}{h^2},
$$
with $h = 1/n$ and periodic boundary conditions (indices mod $n$).

Time integration uses an explicit Euler step with timestep $\tau$. CFL stability requires
$\tau \|u\|_\infty / h \leq 1$ for the transport part. In practice we set $\tau = c\,\nu / n^2$ with a small constant $c$, which is conservative for the diffusion term.

In [ ]:
def burgers_solve(u0, nu=0.01, T=1.5, q=40, n=512):
    """Integrate viscous Burgers on [0,1) periodic, returning q snapshots."""
    tau = nu * 0.4 / n**2          # stable time-step for diffusion
    tau = min(tau, 0.3 / n)        # also respect transport CFL loosely
    niter = max(1, int(T / tau))
    disp  = set(np.round(np.linspace(0, niter - 1, q)).astype(int))

    def dx(f):   return n * (np.roll(f, -1) - np.roll(f,  1)) / 2
    def ddx(f):  return n**2 * (np.roll(f, -1) + np.roll(f, 1) - 2 * f)

    u = u0.copy()
    snaps = []
    for i in range(niter):
        u = u + tau * (-dx(u**2) / 2 + nu * ddx(u))
        if i in disp:
            snaps.append(u.copy())
    return np.array(snaps)


n = 512
x = np.linspace(0, 1, n, endpoint=False)

# Three canonical initial conditions
u0_gauss = np.exp(-(x - 0.25)**2 / (2 * 0.05**2))
u0_sine  = np.sin(2 * np.pi * x)
u0_box   = (np.abs(x - 0.3) < 0.06).astype(float)

print("Grid size:", n, " | time-stepping ready.")

## Shock formation: Gaussian initial condition

A smooth, localized initial profile (here a Gaussian) immediately begins to steepen on its front face: points at the peak travel faster and overtake points ahead. For $\nu = 0$ this creates a **gradient blow-up** in finite time $t^* = \min_x(-1/\partial_x u_0(x))^{-1}$. A small viscosity $\nu \ll 1$ prevents the singularity but leaves a sharp shock layer.

The 40 temporal snapshots transition from blue (early) to red (late).

In [ ]:
nu_demo = 0.004
snaps_g = burgers_solve(u0_gauss, nu=nu_demo, T=1.8, q=40)

fig, ax = plt.subplots(figsize=(8, 4))
for j, u in enumerate(snaps_g):
    s = j / (len(snaps_g) - 1)
    ax.plot(x, u, color=(s, 0, 1 - s), lw=1.5, alpha=0.75)
ax.plot(x, u0_gauss, "k--", lw=1.8, label="initial $u_0$")
ax.set_xlim(0, 1); ax.set_ylim(-0.15, 1.15)
ax.set_xlabel("$x$"); ax.set_ylabel("$u(x,t)$")
ax.set_title(f"Viscous Burgers: Gaussian IC, $\\nu = {nu_demo}$")
ax.legend()
plt.tight_layout()
plt.show()

## Three initial conditions side by side

Different initial profiles produce qualitatively different shock structures:
- **Gaussian**: a single shock travels and diffuses.
- **Sine wave** $\sin(2\pi x)$: steepens symmetrically and folds into a sawtooth-like profile.
- **Box pulse**: develops two shocks (front and back) that eventually merge.

In [ ]:
configs = [
    ("Gaussian",   u0_gauss, 0.004, 1.8),
    ("Sine",       u0_sine,  0.004, 1.2),
    ("Box pulse",  u0_box,   0.004, 1.8),
]

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, (name, u0, nu, T) in zip(axes, configs):
    snaps = burgers_solve(u0, nu=nu, T=T, q=35)
    for j, u in enumerate(snaps):
        s = j / (len(snaps) - 1)
        ax.plot(x, u, color=(s, 0, 1 - s), lw=1.2, alpha=0.7)
    ax.plot(x, u0, "k--", lw=1.5)
    ax.set_xlim(0, 1)
    ax.set_title(name, fontsize=11)
    ax.set_xlabel("$x$")
    ax.set_ylabel("$u$" if ax is axes[0] else "")
fig.suptitle(f"Burgers ($\\nu = 0.004$): three initial conditions", y=1.02)
plt.tight_layout()
plt.show()

## Role of viscosity $\nu$

The viscosity parameter $\nu$ controls the **thickness of the shock layer**. By the **Cole–Hopf transformation**,
$$
u = -2\nu\,\frac{\partial_x \theta}{\theta}, \qquad \theta(x,t) = \int e^{-F_0(y)/(2\nu)} \cdot \frac{1}{\sqrt{4\pi\nu t}} e^{-(x-y)^2/(4\nu t)}\,dy,
$$
the viscous Burgers equation reduces to the **heat equation** for $\theta$. As $\nu \to 0$, the saddle-point approximation of the integral concentrates at $y^*$, recovering the Hopf–Lax entropy solution.

Large $\nu$ makes the shock diffuse rapidly; small $\nu$ preserves a steep but smooth front.

In [ ]:
nu_vals = [0.0008, 0.004, 0.02, 0.1]

fig, axes = plt.subplots(1, 4, figsize=(13, 3.6))
for ax, nu in zip(axes, nu_vals):
    snaps = burgers_solve(u0_gauss, nu=nu, T=1.8, q=30)
    for j, u in enumerate(snaps):
        s = j / (len(snaps) - 1)
        ax.plot(x, u, color=(s, 0, 1 - s), lw=1.2, alpha=0.75)
    ax.plot(x, u0_gauss, "k--", lw=1.4)
    ax.set_xlim(0, 1); ax.set_ylim(-0.1, 1.2)
    ax.set_title(f"$\\nu = {nu}$", fontsize=10)
    ax.set_xlabel("$x$")
    ax.set_ylabel("$u$" if ax is axes[0] else "")
fig.suptitle("Effect of viscosity on shock thickness (Gaussian IC)", y=1.02)
plt.tight_layout()
plt.show()

## Space–time diagram

Encoding the solution amplitude in color produces a **space–time plot** that makes the shock trajectory visible as a line of rapid color transition. The slope of this line is the shock speed, which equals the average of the solution values on both sides of the shock by the **Rankine–Hugoniot condition**:
$$
s = \frac{u_L + u_R}{2}.
$$

In [ ]:
q_st = 120
snaps_st = burgers_solve(u0_gauss, nu=0.004, T=2.0, q=q_st)
U = np.array(snaps_st)        # shape (q_st, n)

fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(U, origin="lower", aspect="auto",
               extent=[0, 1, 0, 2.0], cmap="RdBu_r",
               vmin=-0.2, vmax=1.0)
plt.colorbar(im, ax=ax, label="$u(x,t)$", fraction=0.046)
ax.set_xlabel("$x$"); ax.set_ylabel("$t$")
ax.set_title("Space–time diagram: shock trajectory (Gaussian IC, $\\nu=0.004$)")
plt.tight_layout()
plt.show()

## Interactive exploration

Use the controls to change the initial condition shape and the viscosity. Observe how the shock sharpens as $\nu \to 0$ and how different profiles produce qualitatively different wave structures.

In [ ]:
ic_dict = {
    "Gaussian":  u0_gauss,
    "Sine wave": u0_sine,
    "Box pulse": u0_box,
}

def show_burgers(ic="Gaussian", log_nu=-2.0):
    nu = 10 ** log_nu
    u0 = ic_dict[ic]
    snaps = burgers_solve(u0, nu=nu, T=1.8, q=35)

    fig, ax = plt.subplots(figsize=(8, 4))
    for j, u in enumerate(snaps):
        s = j / (len(snaps) - 1)
        ax.plot(x, u, color=(s, 0, 1 - s), lw=1.4, alpha=0.7)
    ax.plot(x, u0, "k--", lw=1.8, label="$u_0$")
    ax.set_xlim(0, 1)
    ax.set_xlabel("$x$"); ax.set_ylabel("$u(x,t)$")
    ax.set_title(f"{ic} IC  |  $\\nu = {nu:.4f}$")
    ax.legend()
    plt.tight_layout(); plt.show()

interact(
    show_burgers,
    ic=Dropdown(options=list(ic_dict.keys()), description="IC"),
    log_nu=FloatLogSlider(value=-2.0, base=10, min=-3.5, max=-0.5, step=0.25,
                          description="$\\log_{10}\\nu$"),
);

## Entropy and energy decay

For smooth solutions the $L^2$ energy satisfies
$$
\frac{d}{dt}\int_0^1 u^2\,dx = -2\nu \int_0^1 (\partial_x u)^2\,dx \leq 0.
$$

The dissipation rate $\epsilon(t) = 2\nu\int(\partial_x u)^2\,dx$ peaks when the gradient is steepest (near shock formation) and then decays as diffusion smooths the front. The plot below compares energy curves for three viscosity values.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
T_en = 3.0
for nu_e in [0.001, 0.01, 0.05]:
    q_e = 80
    snaps_e = burgers_solve(u0_gauss, nu=nu_e, T=T_en, q=q_e)
    times = np.linspace(0, T_en, q_e)

    energy = np.array([np.mean(u**2) for u in snaps_e])
    grad   = np.array([
        2 * nu_e * np.mean((n * np.diff(u, append=u[0]))**2)
        for u in snaps_e
    ])
    lbl = f"$\\nu={nu_e}$"
    axes[0].plot(times, energy, lw=2, label=lbl)
    axes[1].plot(times, grad,   lw=2, label=lbl)

axes[0].set_xlabel("$t$"); axes[0].set_ylabel(r"$\|u\|_2^2$")
axes[0].set_title("$L^2$ energy")
axes[0].legend()
axes[1].set_xlabel("$t$")
axes[1].set_title(r"Dissipation $2\nu\|\partial_x u\|_2^2$")
axes[1].legend()
plt.tight_layout()
plt.show()

## Bibliographical resources

- Burgers, J. M. (1948). A mathematical model illustrating the theory of turbulence. *Advances in Applied Mechanics*, 1, 171–199.
- Hopf, E. (1950). The partial differential equation $u_t + uu_x = \mu u_{xx}$. *Communications on Pure and Applied Mathematics*, 3(3), 201–230.
- Cole, J. D. (1951). On a quasi-linear parabolic equation occurring in aerodynamics. *Quarterly of Applied Mathematics*, 9(3), 225–236.
- Evans, L. C. (2010). *Partial Differential Equations* (2nd ed.). AMS Graduate Studies in Mathematics.
- LeVeque, R. J. (2002). *Finite Volume Methods for Hyperbolic Problems*. Cambridge University Press.
- Serre, D. (1999). *Systems of Conservation Laws* (Vol. 1). Cambridge University Press.